# COS40007 Artificial Intelligence Engineering
## Portfolio Assessment 3: Advanced ML – Classification using ML and Deep Learning
### Option 3c – LSTM for Classification

| | |
|---|---|
| **Student name** | Dinh Phu Nguyen |
| **Student ID** | 105027282 |
| **Date** | 11/1/2026 |
| **Dataset** | Appliances Energy Prediction – Candanedo, L. (2017), UCI Machine Learning Repository, DOI: [10.24432/C5VC8G](https://doi.org/10.24432/C5VC8G) |
| **GitHub** | [https://github.com/phu1009/COS40007---AI-Engineering-] |

**GenAI declaration:** I used Claude (Anthropic) in choosing option 3c, analysing the instructions and rubric and suggest the plan to do the Assignment, and organising and formatting the report. The full GenAI reflection (Part E) is in the PDF report.

## Problem definition and choice of option 3c
The dataset (cleaned in PA1, used for classification in PA2) contains **19,735 readings taken every 10 minutes** (11 January – 27 May 2016) of the appliance energy use of a low-energy house, together with indoor temperature and humidity sensors and outdoor weather data. The target is the **consumption class** defined in PA2 Task 1:

| Class | Appliances (Wh per 10 min) | Meaning (PA1 §3.3) |
|---|---|---|
| Low (0) | ≤ 50 | night-time base load |
| Medium (1) | 60–80 | light daytime activity |
| High (2) | ≥ 90 | high-power appliances, e.g. cooking in the evening peak |

The data is a **real time series** (sensor data) and the target is **categorical**, so the selection table of the instructions points to **option 3c – LSTM for Classification**.

### Design decisions kept fixed for Part A and Part B
| Decision | Choice | Reason |
|---|---|---|
| Features | 24 cleaned predictors + `hour_sin`, `hour_cos` (26) | the sensors are what the house measures; the time encoding captures the daily routine (PA2 Task 3: `hour_sin` was the most informative feature). No hand-made "change over time" features: learning temporal patterns from raw sequences is the job of the LSTM, so Part A vs Part B measures the value of temporal context |
| Split | **chronological 70/30** (train 11 Jan – 16 Apr, test 16 Apr – 27 May) | PA2 Task 10.2 showed that shuffled splits are optimistic for this time series; LSTM windows overlap, so a shuffled split would put near-identical windows in both sets |
| Test rows | the **same 5,921 test readings** for every model | fair comparison in Part C |

## 0. Setup

In [1]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"          # hide TensorFlow C++ info / warning messages

import time
import pickle
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sklearn
from sklearn.base import clone
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, TimeSeriesSplit, cross_val_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, f1_score, log_loss, classification_report,
                             ConfusionMatrixDisplay)
import tensorflow as tf
import keras
from keras import layers
from IPython.display import display

RANDOM_STATE = 42
TEST_SIZE = 0.30                     # chronological 70/30 split
N_FOLDS = 5                          # TimeSeriesSplit folds for Part A
CLASS_LABELS = ["Low", "Medium", "High"]
DATA_PATH = Path("data") / "energydata_complete.csv"
FIG_DIR, MODEL_DIR = Path("figures"), Path("models")
FIG_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)

# Reproducibility: same seed for numpy, Python and TensorFlow, deterministic TensorFlow operations
keras.utils.set_random_seed(RANDOM_STATE)
tf.config.experimental.enable_op_determinism()

sns.set_theme(style="whitegrid", context="notebook")
COLOR_MAIN, COLOR_SECOND, COLOR_THIRD, COLOR_GREY, COLOR_TEXT = "#2A78D6", "#E8683A", "#3A9D5D", "#B5B5AD", "#444444"
pd.set_option("display.precision", 4)


def save_fig(fig, name):
    """Save a figure for the report."""
    fig.savefig(FIG_DIR / f"{name}.png", dpi=200, bbox_inches="tight")


print(f"Python {platform.python_version()} | pandas {pd.__version__} | numpy {np.__version__} | "
      f"scikit-learn {sklearn.__version__} | TensorFlow {tf.__version__} | Keras {keras.__version__}")

Python 3.13.16 | pandas 3.0.5 | numpy 2.5.3 | scikit-learn 1.9.1 | TensorFlow 2.21.0 | Keras 3.15.1
